# AlterNet 2.0 - Network Inference (GTEx)

This notebook runs GRNBoost2 inference for three networks:
1. **Network 1 (Canonical)**: TF Gene → Target Gene
2. **Network 2 (AS-Aware Source)**: TF Transcript → Target Gene
3. **Network 3 (Fully AS-Aware)**: TF+SF Transcript → Target Transcript

## Setup and Imports

In [1]:
import sys
sys.path.append('/Users/zihengdai/Desktop/Thesis/data/')

import pandas as pd
import numpy as np
import yaml
import time
import os
import os.path as op

# AlterNet imports
from alternet.data_preprocessing import create_hybrid_data, standardize_dataframe
from alternet.annotation import map_tf_ids, create_transcript_mapping
from alternet.annotation import create_filtered_gene_to_transcripts_mapping
from alternet.annotation import create_transcipt_annotation_database
import alternet.annotation as annotation
import alternet.postprocessing as postprocessing
from alternet.inference import inference

## Configuration

In [2]:
data_path = "./"
results_path = "./results_gtex_net_infer/"

# Reference files
appris_path = "appris_data.appris.txt"
digger_path = "digger_data.csv"
biomart_path = "biomart.txt"
tf_list_path = "allTFs_hg38.txt"
sf_list_path = "comprehensive_sfs.csv"

# Expression data
gtex_transcript_tpm_path = "GTEx_Analysis_v10_RSEMv1.3.3_transcripts_tpm.txt"
gtex_sample_attributes_path = "GTEx_Analysis_v10_Annotations_SampleAttributesDS.txt"

# Tissue to analyze
TISSUE = "Bladder"
CONDITION = TISSUE

# Number of GRNBoost2 runs
N_RUNS = 10

os.makedirs(results_path, exist_ok=True)

## Helper Functions

In [3]:
def write_dict_to_yaml(data, filepath):
    """Write dictionary to YAML file."""
    with open(filepath, 'w') as f:
        yaml.dump(data, f, default_flow_style=False)

def map_sf_ids(sf_list_raw, biomart):
    """Map SF gene names to gene and transcript IDs."""
    sf_list_raw = sf_list_raw.copy()
    sf_list_raw.columns = ['SF']
    sf_list = sf_list_raw.merge(biomart, left_on='SF', right_on='Gene name')
    sf_list = sf_list.loc[:, ['SF', 'Gene stable ID', 'Transcript stable ID']].drop_duplicates()
    return sf_list

def combine_tf_sf_lists(tf_list, sf_list):
    """Combine TF and SF lists, marking overlapping genes as TF_SF."""
    tf_list = tf_list.copy()
    sf_list = sf_list.copy()
    
    tf_list['Regulator_type'] = 'TF'
    sf_list['Regulator_type'] = 'SF'
    tf_list = tf_list.rename(columns={'TF': 'Regulator_name'})
    sf_list = sf_list.rename(columns={'SF': 'Regulator_name'})
    
    tf_genes = set(tf_list['Gene stable ID'])
    sf_genes = set(sf_list['Gene stable ID'])
    overlap_genes = tf_genes & sf_genes
    
    print(f"TF genes: {len(tf_genes)}")
    print(f"SF genes: {len(sf_genes)}")
    print(f"Overlap (TF+SF): {len(overlap_genes)}")
    
    combined = pd.concat([tf_list, sf_list], ignore_index=True)
    combined.loc[combined['Gene stable ID'].isin(overlap_genes), 'Regulator_type'] = 'TF_SF'
    combined = combined.drop_duplicates(subset=['Transcript stable ID'], keep='first')
    
    return combined

## Load Reference Data

In [4]:
# Load reference files
biomart = pd.read_csv(biomart_path, sep='\t')
appris_df = pd.read_csv(appris_path, sep='\t')
digger_df = pd.read_csv(digger_path, low_memory=False)

print(f"BioMart entries: {len(biomart)}")
print(f"APPRIS entries: {len(appris_df)}")
print(f"DIGGER entries: {len(digger_df)}")

BioMart entries: 278220
APPRIS entries: 170712
DIGGER entries: 944451


In [5]:
# Load and map TF list
tf_list_raw = pd.read_csv(tf_list_path, sep='\t', header=None)
tf_list = map_tf_ids(tf_list_raw, biomart)

print(f"Unique TF genes: {tf_list['Gene stable ID'].nunique()}")
print(f"Unique TF transcripts: {tf_list['Transcript stable ID'].nunique()}")

Unique TF genes: 1948
Unique TF transcripts: 16298


In [6]:
# Load and map SF list
sf_list_raw = pd.read_csv(sf_list_path, header=None)
sf_list = map_sf_ids(sf_list_raw, biomart)

print(f"Unique SF genes: {sf_list['Gene stable ID'].nunique()}")
print(f"Unique SF transcripts: {sf_list['Transcript stable ID'].nunique()}")

Unique SF genes: 264
Unique SF transcripts: 3227


In [7]:
# Combine TF and SF lists
regulator_list = combine_tf_sf_lists(tf_list, sf_list)

print(f"\nTotal unique genes: {regulator_list['Gene stable ID'].nunique()}")
print(f"Total unique transcripts: {regulator_list['Transcript stable ID'].nunique()}")
print(f"\nBy regulator type:")
print(regulator_list['Regulator_type'].value_counts())

TF genes: 1948
SF genes: 264
Overlap (TF+SF): 42

Total unique genes: 2170
Total unique transcripts: 18958

By regulator type:
Regulator_type
TF       15731
SF        2660
TF_SF      567
Name: count, dtype: int64


In [8]:
# Create mappings
transcript_mapper = annotation.create_transcript_mapping(biomart)
print(f"Transcript-to-gene mappings: {len(transcript_mapper)}")

# Annotation databases
tf_database = annotation.create_transcipt_annotation_database(
    tf_list=tf_list, appris_df=appris_df, digger=digger_df
)
regulator_database = annotation.create_transcipt_annotation_database(
    tf_list=regulator_list, appris_df=appris_df, digger=digger_df
)
print(f"TF annotation database: {len(tf_database)} entries")
print(f"Regulator annotation database: {len(regulator_database)} entries")

Transcript-to-gene mappings: 278220
TF annotation database: 16298 entries
Regulator annotation database: 18958 entries


## Load Expression Data

In [9]:
# Load sample attributes
sample_attributes = pd.read_csv(gtex_sample_attributes_path, sep='\t', low_memory=False)
print(f"Total samples in GTEx: {len(sample_attributes)}")

# Get tissue samples
tissue_samples = sample_attributes[sample_attributes['SMTSD'] == TISSUE]['SAMPID'].tolist()
print(f"Samples for {TISSUE}: {len(tissue_samples)}")

Total samples in GTEx: 48231
Samples for Bladder: 134


In [10]:
# Get column names from expression file
header_df = pd.read_csv(gtex_transcript_tpm_path, sep='\t', nrows=0)
all_columns = header_df.columns.tolist()
id_col_1, id_col_2 = all_columns[0], all_columns[1]

# Find matching samples
sample_columns_in_file = all_columns[2:]
matching_samples = [c for c in sample_columns_in_file if c in tissue_samples]
print(f"Matching samples in expression file: {len(matching_samples)}")

# Load expression data
columns_to_read = [id_col_1, id_col_2] + matching_samples
transcript_data_raw = pd.read_csv(gtex_transcript_tpm_path, sep='\t', usecols=columns_to_read)
print(f"Loaded: {transcript_data_raw.shape[0]} transcripts × {len(matching_samples)} samples")

Matching samples in expression file: 77
Loaded: 244939 transcripts × 77 samples


## Preprocess Expression Data

In [11]:
# Standardize column names and remove version numbers
transcript_data = transcript_data_raw.copy()
transcript_data = transcript_data.rename(columns={id_col_1: 'transcript_id', id_col_2: 'gene_id'})

transcript_data['transcript_id'] = transcript_data['transcript_id'].str.split('.').str[0]
transcript_data['gene_id'] = transcript_data['gene_id'].str.split('.').str[0]

sample_cols = [c for c in transcript_data.columns if c not in ['transcript_id', 'gene_id']]
print(f"Transcripts: {len(transcript_data)}, Samples: {len(sample_cols)}")

Transcripts: 244939, Samples: 77


In [12]:
# Filter to protein-coding transcripts
protein_coding = biomart[biomart['Gene type'] == 'protein_coding']['Transcript stable ID'].unique()
n_before = len(transcript_data)
transcript_data = transcript_data[transcript_data['transcript_id'].isin(protein_coding)].copy()
print(f"Protein-coding filter: {n_before} → {len(transcript_data)} transcripts")

Protein-coding filter: 244939 → 163586 transcripts


In [13]:
# Remove low-expression transcripts
threshold = len(sample_cols) * 0.1
zero_counts = (transcript_data[sample_cols] == 0).sum(axis=1)
n_before = len(transcript_data)
transcript_data = transcript_data[zero_counts < threshold].copy()
print(f"Low-expression filter: {n_before} → {len(transcript_data)} transcripts")
print(f"Unique genes: {transcript_data['gene_id'].nunique()}")

Low-expression filter: 163586 → 55777 transcripts
Unique genes: 14843


In [14]:
VARIANCE_PERCENTILE = 0.7  # Keep top 30%

expression_values = transcript_data[sample_cols].values
log_expr = np.log1p(expression_values)
variances = np.var(log_expr, axis=1)
variance_threshold = np.quantile(variances, VARIANCE_PERCENTILE)

n_before = len(transcript_data)
transcript_data = transcript_data[variances > variance_threshold].copy()
print(f"Variance filter: {n_before} → {len(transcript_data)} transcripts")

Variance filter: 55777 → 16733 transcripts


In [15]:
# Create gene-level data
gene_data = transcript_data.groupby('gene_id')[sample_cols].sum().reset_index()
print(f"Gene-level data: {len(gene_data)} genes")

Gene-level data: 7577 genes


In [16]:
# Create expression matrices (samples × features)
gene_data_matrix = gene_data.set_index('gene_id')[sample_cols].T
transcript_data_matrix = transcript_data.set_index('transcript_id')[sample_cols].T

print(f"Gene matrix: {gene_data_matrix.shape} (samples × genes)")
print(f"Transcript matrix: {transcript_data_matrix.shape} (samples × transcripts)")

# Standardize (z-score)
gene_data_scaled = standardize_dataframe(gene_data_matrix)
transcript_data_scaled = standardize_dataframe(transcript_data_matrix)

Gene matrix: (77, 7577) (samples × genes)
Transcript matrix: (77, 16733) (samples × transcripts)


In [17]:
# Remove problematic transcripts (NaN or zero variance)
nan_cols = transcript_data_scaled.columns[transcript_data_scaled.isna().any()].tolist()
zero_var_cols = transcript_data_matrix.columns[transcript_data_matrix.std() == 0].tolist()
bad_transcripts = set(nan_cols + zero_var_cols)

if bad_transcripts:
    good_transcripts = [c for c in transcript_data_scaled.columns if c not in bad_transcripts]
    transcript_data_scaled = transcript_data_scaled[good_transcripts]
    transcript_data_matrix = transcript_data_matrix[good_transcripts]
    print(f"Removed {len(bad_transcripts)} problematic transcripts")

# Fill remaining NaN
transcript_data_scaled = transcript_data_scaled.fillna(0)
gene_data_scaled = gene_data_scaled.fillna(0)

print(f"Final: {len(transcript_data_scaled.columns)} transcripts, {len(gene_data_scaled.columns)} genes")

Final: 16733 transcripts, 7577 genes


## Identify Regulators in Data

In [18]:
# Gene-to-transcript mapping for genes in data
gene_to_transcript_mapping = annotation.create_filtered_gene_to_transcripts_mapping(
    biomart,
    gene_list=gene_data_scaled.columns,
    transcript_list=transcript_data_scaled.columns
)
print(f"Gene-to-transcript mappings: {len(gene_to_transcript_mapping)}")

Gene-to-transcript mappings: 7566


In [19]:
# TF genes in data (for Network 1)
tf_genes_in_data = list(set(tf_list['Gene stable ID']) & set(gene_data_scaled.columns))
print(f"TF genes in data: {len(tf_genes_in_data)}")

# TF transcripts in data (for Network 2)
tf_transcripts_in_data = list(set(tf_list['Transcript stable ID']) & set(transcript_data_scaled.columns))
print(f"TF transcripts in data: {len(tf_transcripts_in_data)}")

# All regulator transcripts (for Network 3)
regulator_transcripts_in_data = list(
    set(regulator_list['Transcript stable ID']) & set(transcript_data_scaled.columns)
)
print(f"All regulator transcripts (TF+SF) in data: {len(regulator_transcripts_in_data)}")

# Targets
target_genes = list(gene_data_scaled.columns)
target_transcripts = list(transcript_data_scaled.columns)
print(f"Target genes: {len(target_genes)}")
print(f"Target transcripts: {len(target_transcripts)}")

TF genes in data: 652
TF transcripts in data: 1421
All regulator transcripts (TF+SF) in data: 1817
Target genes: 7577
Target transcripts: 16733


In [20]:
# Regulator types in data
tf_only_transcripts = set(regulator_list[regulator_list['Regulator_type'] == 'TF']['Transcript stable ID'])
sf_only_transcripts = set(regulator_list[regulator_list['Regulator_type'] == 'SF']['Transcript stable ID'])
tfsf_transcripts = set(regulator_list[regulator_list['Regulator_type'] == 'TF_SF']['Transcript stable ID'])

tf_only_transcripts = tf_only_transcripts & set(transcript_data_scaled.columns)
sf_only_transcripts = sf_only_transcripts & set(transcript_data_scaled.columns)
tfsf_transcripts = tfsf_transcripts & set(transcript_data_scaled.columns)

print(f"TF only: {len(tf_only_transcripts)}")
print(f"SF only: {len(sf_only_transcripts)}")
print(f"TF+SF: {len(tfsf_transcripts)}")

TF only: 1321
SF only: 396
TF+SF: 100


## Compute Isoform Categories

In [21]:
# TF isoform categories (for Network 1 & 2 comparison)
tf_isoform_categories = postprocessing.isoform_categorization(
    transcript_data_matrix, gene_data_matrix, tf_list
)
tf_gene_categories = postprocessing.get_gene_cases(tf_isoform_categories)

print("TF isoform categories:")
print(tf_isoform_categories['isoform_category'].value_counts())

TF isoform categories:
isoform_category
balanced        529
non-dominant    523
single          340
dominant         29
Name: count, dtype: int64


In [22]:
# Regulator isoform categories (for Network 3)
regulator_isoform_categories = postprocessing.isoform_categorization(
    transcript_data_matrix, gene_data_matrix, regulator_list
)
regulator_gene_categories = postprocessing.get_gene_cases(regulator_isoform_categories)

print("Regulator isoform categories:")
print(regulator_isoform_categories['isoform_category'].value_counts())

Regulator isoform categories:
isoform_category
balanced        767
non-dominant    632
single          381
dominant         36
Name: count, dtype: int64


In [23]:
# Target isoform categories
target_list = biomart[
    biomart['Transcript stable ID'].isin(transcript_data_scaled.columns)
][['Gene stable ID', 'Transcript stable ID']].drop_duplicates()

target_isoform_categories = postprocessing.isoform_categorization(
    transcript_data_matrix, gene_data_matrix, target_list
)
target_gene_categories = postprocessing.get_gene_cases(target_isoform_categories)

print("Target isoform categories:")
print(target_isoform_categories['isoform_category'].value_counts())

Target isoform categories:
isoform_category
non-dominant    6743
balanced        5670
single          3773
dominant         532
Name: count, dtype: int64


In [24]:
# Create category lookups
tf_transcript_to_category = dict(zip(
    tf_isoform_categories['Transcript stable ID'], tf_isoform_categories['isoform_category']
))
regulator_transcript_to_category = dict(zip(
    regulator_isoform_categories['Transcript stable ID'], regulator_isoform_categories['isoform_category']
))
target_transcript_to_category = dict(zip(
    target_isoform_categories['Transcript stable ID'], target_isoform_categories['isoform_category']
))

tf_gene_to_category = dict(zip(
    tf_gene_categories['Gene stable ID'], tf_gene_categories['gene_category']
))
regulator_gene_to_category = dict(zip(
    regulator_gene_categories['Gene stable ID'], regulator_gene_categories['gene_category']
))
target_gene_to_category = dict(zip(
    target_gene_categories['Gene stable ID'], target_gene_categories['gene_category']
))

## Run Network Inference

In [25]:
runtime = {}

In [26]:
# NETWORK 1: Canonical
print("NETWORK 1: Canonical")
print(f"Regulators: {len(tf_genes_in_data)} TF genes")
print(f"Targets: {len(target_genes)} genes")

start = time.monotonic()
canonical_grn = inference(
    gene_data=gene_data_scaled,
    tf_list=tf_genes_in_data,
    target_names='all',
    n_runs=N_RUNS
)
runtime['canonical'] = time.monotonic() - start

print(f"\nEdges: {len(canonical_grn):,}")
print(f"Time: {runtime['canonical']/60:.2f} minutes")

# Save
canonical_grn.to_csv(op.join(results_path, f"{CONDITION}_canonical_raw.tsv"), sep='\t', index=False)

NETWORK 1: Canonical
Regulators: 652 TF genes
Targets: 7577 genes


100%|██████████████████████████████████████████| 10/10 [40:30<00:00, 243.09s/it]



Edges: 3,609,682
Time: 40.78 minutes
Saved: Bladder_canonical_raw.tsv


In [27]:
# NETWORK 2: AS-Aware Source
print("NETWORK 2: AS-Aware Source")
print(f"Regulators: {len(tf_transcripts_in_data)} TF transcripts")
print(f"Targets: {len(target_genes)} genes")

# Create hybrid data (TF transcripts + target genes)
hybrid_data = create_hybrid_data(
    transcript_data_matrix,  
    gene_data_matrix,        
    tf_list
)
print(f"Hybrid data shape: {hybrid_data.shape}")

start = time.monotonic()
as_source_grn = inference(
    gene_data=hybrid_data,
    tf_list=tf_transcripts_in_data,
    target_names=target_genes,
    n_runs=N_RUNS
)
runtime['as_aware_source'] = time.monotonic() - start

print(f"\nEdges: {len(as_source_grn):,}")
print(f"Time: {runtime['as_aware_source']/60:.2f} minutes")

# Save
as_source_grn.to_csv(op.join(results_path, f"{CONDITION}_as_aware_source_raw.tsv"), sep='\t', index=False)

NETWORK 2: AS-Aware Source
Regulators: 1421 TF transcripts
Targets: 7577 genes
Hybrid data shape: (77, 8998)


/home/hpc/iwbn/iwbn121h/alternet2_env/lib/python3.12/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44315 instead
  warnings.warn(
100%|██████████████████████████████████████████| 10/10 [49:28<00:00, 296.89s/it]



Edges: 5,264,965
Time: 49.70 minutes
Saved: Bladder_as_aware_source_raw.tsv


In [ ]:
# NETWORK 3: Fully AS-Aware
print("NETWORK 3: Fully AS-Aware")
print(f"Regulators: {len(regulator_transcripts_in_data)} transcripts (TF+SF)")
print(f"Targets: {len(target_transcripts)} transcripts")

start = time.monotonic()
fully_as_grn = inference(
    gene_data=transcript_data_scaled,
    tf_list=regulator_transcripts_in_data,
    target_names='all',
    n_runs=N_RUNS
)
runtime['fully_as_aware'] = time.monotonic() - start

print(f"\nEdges: {len(fully_as_grn):,}")
print(f"Time: {runtime['fully_as_aware']/60:.2f} minutes")

# Save
fully_as_grn.to_csv(op.join(results_path, f"{CONDITION}_fully_as_aware_raw.tsv"), sep='\t', index=False)

NETWORK 3: Fully AS-Aware
Regulators: 1817 transcripts (TF+SF)
Targets: 16733 transcripts


/home/hpc/iwbn/iwbn121h/alternet2_env/lib/python3.12/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44531 instead
  warnings.warn(
  0%|                                                    | 0/10 [00:00<?, ?it/s]/home/hpc/iwbn/iwbn121h/alternet2_env/lib/python3.12/site-packages/distributed/client.py:3374: UserWarning: Sending large graph of size 16.85 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [ ]:
# Save runtime
runtime['total'] = runtime['canonical'] + runtime['as_aware_source'] + runtime['fully_as_aware']
write_dict_to_yaml(runtime, op.join(results_path, f"{CONDITION}_runtime.yaml"))
print(f"\nTotal inference time: {runtime['total']/60:.2f} minutes")

## Save Metadata

In [ ]:
# Save isoform categories
tf_isoform_categories.to_csv(
    op.join(results_path, f"{CONDITION}_tf_isoform_categories.csv"), index=False
)
regulator_isoform_categories.to_csv(
    op.join(results_path, f"{CONDITION}_regulator_isoform_categories.csv"), index=False
)
target_isoform_categories.to_csv(
    op.join(results_path, f"{CONDITION}_target_isoform_categories.csv"), index=False
)

# Save gene categories
tf_gene_categories.to_csv(
    op.join(results_path, f"{CONDITION}_tf_gene_categories.csv"), index=False
)
regulator_gene_categories.to_csv(
    op.join(results_path, f"{CONDITION}_regulator_gene_categories.csv"), index=False
)
target_gene_categories.to_csv(
    op.join(results_path, f"{CONDITION}_target_gene_categories.csv"), index=False
)

# Save regulator list with types
regulator_list.to_csv(
    op.join(results_path, f"{CONDITION}_regulator_list.csv"), index=False
)

In [ ]:
# Save summary statistics
summary_stats = {
    'tissue': TISSUE,
    'n_samples': len(sample_cols),
    'n_genes': len(gene_data_scaled.columns),
    'n_transcripts': len(transcript_data_scaled.columns),
    'n_tf_genes': len(tf_genes_in_data),
    'n_tf_transcripts': len(tf_transcripts_in_data),
    'n_sf_transcripts': len(sf_only_transcripts),
    'n_tfsf_transcripts': len(tfsf_transcripts),
    'n_regulator_transcripts': len(regulator_transcripts_in_data),
    'network1_edges': len(canonical_grn),
    'network2_edges': len(as_source_grn),
    'network3_edges': len(fully_as_grn),
    'runtime_canonical_min': runtime['canonical'] / 60,
    'runtime_as_source_min': runtime['as_aware_source'] / 60,
    'runtime_fully_as_min': runtime['fully_as_aware'] / 60,
    'runtime_total_min': runtime['total'] / 60,
}

write_dict_to_yaml(summary_stats, op.join(results_path, f"{CONDITION}_summary_stats.yaml"))